In [3]:
%pip install -q importlib_resources

In [4]:

# SWE015 - Step 1: Load EMNIST Balanced
!pip -q install tensorflow-datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

print("TensorFlow version:", tf.__version__)

# Load EMNIST Balanced using TensorFlow Datasets
train_ds, test_ds = tfds.load(
    "emnist/balanced",
    split=["train", "test"],
    as_supervised=True
)

# Convert to NumPy arrays
def convert_dataset(dataset):
    images = []
    labels = []

    for image, label in tfds.as_numpy(dataset):
        images.append(image)
        labels.append(label)

    images = np.array(images, dtype=np.float32) / 255.0
    labels = np.array(labels, dtype=np.int32)

    return images, labels

x_train, y_train = convert_dataset(train_ds)
x_test, y_test = convert_dataset(test_ds)

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)
print("Number of classes:", len(np.unique(y_train)))


TensorFlow version: 2.21.0


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/balanced/incomplete.VTTYNL_3.1.0/emnist-train.tfrecord-[0-9][0-9][0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/balanced/incomplete.VTTYNL_3.1.0/emnist-test.tfrecord-[0-9][0-9][0-…

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/balanced/3.1.0. Subsequent calls will reuse this data.
Training images: (112800, 28, 28, 1)
Testing images: (18800, 28, 28, 1)
Number of classes: 47


In [5]:

# Step 2: Prepare Train / Validation / Test

from sklearn.model_selection import train_test_split

# Keep the official test set untouched
x_train_final, x_val, y_train_final, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train
)

# Flatten images for the Dense Neural Network
x_train_dense = x_train_final.reshape(-1, 784)
x_val_dense = x_val.reshape(-1, 784)
x_test_dense = x_test.reshape(-1, 784)

print("Training:", x_train_dense.shape)
print("Validation:", x_val_dense.shape)
print("Testing:", x_test_dense.shape)
print("Classes:", len(np.unique(y_train_final)))


Training: (101520, 784)
Validation: (11280, 784)
Testing: (18800, 784)
Classes: 47


In [6]:

# Step 3: Build and Train Dense Neural Network

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

dense_model = Sequential([
    Input(shape=(784,)),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(47, activation="softmax")
])

dense_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

dense_model.summary()

history_dense = dense_model.fit(
    x_train_dense,
    y_train_final,
    validation_data=(x_val_dense, y_val),
    epochs=15,
    batch_size=128,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ],
    verbose=1
)

dense_model.save("swe015_dense_emnist.keras")
print("Dense model saved successfully!")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 47)             │         6,063 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 239,919 (937.18 KB)

 Trainable params: 239,919 (937.18 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.1312 - loss: 3.4743 - val_accuracy: 0.4201 - val_loss: 2.6904
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3493 - loss: 2.4399 - val_accuracy: 0.5496 - val_loss: 1.7985
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.4591 - loss: 1.9339 - val_accuracy: 0.5982 - val_loss: 1.5095
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5136 - loss: 1.7114 - val_accuracy: 0.6261 - val_loss: 1.3649
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.5505 - loss: 1.5720 - val_accuracy: 0.6469 - val_loss: 1.2670
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.5772 - loss: 1.4706 - val_accuracy: 0.6651 - val_loss: 1.1952
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.5978 - loss: 1.3944 - val_accuracy: 0.6816 - val_loss: 1.1335
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.6139 - loss: 1.3322 - val_accuracy: 0.

In [7]:

# Step 4: Evaluate Dense Model

import os
import json
import time
import numpy as np

# Evaluate on untouched test set
test_loss, test_accuracy = dense_model.evaluate(
    x_test_dense,
    y_test,
    verbose=0
)

# Measure model file size
model_path = "swe015_dense_emnist.keras"
model_size_mb = os.path.getsize(model_path) / (1024 ** 2)

# Measure inference latency on one sample
sample = x_test_dense[:1]

# Warm-up
for _ in range(10):
    dense_model(sample, training=False)

times = []

for _ in range(100):
    start = time.perf_counter()
    dense_model(sample, training=False)
    times.append((time.perf_counter() - start) * 1000)

avg_latency = float(np.mean(times))

results = {
    "model": "Dense Neural Network",
    "dataset": "EMNIST Balanced",
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "model_size_mb": float(model_size_mb),
    "avg_latency_ms": avg_latency,
    "latency_runs": 100
}

with open("dense_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Test Accuracy:", f"{test_accuracy * 100:.2f}%")
print("Test Loss:", round(test_loss, 4))
print("Model Size:", f"{model_size_mb:.2f} MB")
print("Average Latency:", f"{avg_latency:.2f} ms")
print("Results saved!")


Test Accuracy: 74.34%
Test Loss: 0.8708
Model Size: 0.94 MB
Average Latency: 10.89 ms
Results saved!


In [8]:

# Step 5: Train CNN on EMNIST Balanced

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    Flatten, Dense, Dropout,
    RandomRotation, RandomTranslation
)
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

cnn_model = Sequential([
    Input(shape=(28, 28, 1)),

    # Data augmentation
    RandomRotation(0.05),
    RandomTranslation(0.05, 0.05),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(47, activation="softmax")
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

history_cnn = cnn_model.fit(
    x_train_final,
    y_train_final,
    validation_data=(x_val, y_val),
    epochs=12,
    batch_size=128,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ],
    verbose=1
)

cnn_model.save("swe015_cnn_emnist.keras")

print("CNN model saved successfully!")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation                 │ (None, 28, 28, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 28, 28, 1)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 47)             │         6,063 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 229,807 (897.68 KB)

 Trainable params: 229,807 (897.68 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 83s 102ms/step - accuracy: 0.5872 - loss: 1.4013 - val_accuracy: 0.8188 - val_loss: 0.5494
Epoch 2/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.7422 - loss: 0.7980 - val_accuracy: 0.8404 - val_loss: 0.4630
Epoch 3/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 80s 101ms/step - accuracy: 0.7754 - loss: 0.6923 - val_accuracy: 0.8479 - val_loss: 0.4433
Epoch 4/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 80s 101ms/step - accuracy: 0.7939 - loss: 0.6263 - val_accuracy: 0.8553 - val_loss: 0.4157
Epoch 5/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 79s 100ms/step - accuracy: 0.8066 - loss: 0.5822 - val_accuracy: 0.8602 - val_loss: 0.3942
Epoch 6/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 79s 99ms/step - accuracy: 0.8160 - loss: 0.5497 - val_accuracy: 0.8605 - val_loss: 0.3968
Epoch 7/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 83s 100ms/step - accuracy: 0.8236 - loss: 0.5279 - val_accuracy: 0.8672 - val_loss: 0.3788
Epoch 8/12
794/794 ━━━━━━━━━━━━━━━━━━━━ 82s 100ms/step - accuracy: 0.8284 - loss: 0.

In [9]:

# Step 6: Evaluate CNN Model

import os
import json
import time
import numpy as np

# Evaluate on the same untouched test set
test_loss_cnn, test_accuracy_cnn = cnn_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

# Model size
cnn_path = "swe015_cnn_emnist.keras"
cnn_size_mb = os.path.getsize(cnn_path) / (1024 ** 2)

# Latency benchmark
sample = x_test[:1]

for _ in range(10):
    cnn_model(sample, training=False)

times = []

for _ in range(100):
    start = time.perf_counter()
    cnn_model(sample, training=False)
    times.append((time.perf_counter() - start) * 1000)

cnn_latency = float(np.mean(times))

cnn_results = {
    "model": "CNN",
    "dataset": "EMNIST Balanced",
    "test_accuracy": float(test_accuracy_cnn),
    "test_loss": float(test_loss_cnn),
    "model_size_mb": float(cnn_size_mb),
    "avg_latency_ms": cnn_latency,
    "latency_runs": 100
}

with open("cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=4)

print("CNN Test Accuracy:", f"{test_accuracy_cnn * 100:.2f}%")
print("CNN Test Loss:", round(test_loss_cnn, 4))
print("CNN Model Size:", f"{cnn_size_mb:.2f} MB")
print("CNN Average Latency:", f"{cnn_latency:.2f} ms")
print("CNN results saved successfully!")


CNN Test Accuracy: 87.45%
CNN Test Loss: 0.3636
CNN Model Size: 2.67 MB
CNN Average Latency: 10.04 ms
CNN results saved successfully!


In [10]:

# Step 7: Back up trained models and results

from google.colab import drive
from pathlib import Path
import shutil
import platform
import tensorflow as tf

drive.mount("/content/drive")

backup_dir = Path(
    "/content/drive/MyDrive/SWE015_Handwriting_Project/models_and_results"
)
backup_dir.mkdir(parents=True, exist_ok=True)

files_to_save = [
    "swe015_dense_emnist.keras",
    "dense_results.json",
    "swe015_cnn_emnist.keras",
    "cnn_results.json"
]

for filename in files_to_save:
    source = Path("/content") / filename

    if source.exists():
        shutil.copy2(source, backup_dir / filename)
        print("Saved:", filename)
    else:
        print("MISSING:", filename)

print("\nHardware:", platform.processor())
print("CPU:", platform.machine())
print("GPU:", tf.config.list_physical_devices("GPU"))
print("\nBackup folder:", backup_dir)


MessageError: Error: credential propagation was unsuccessful

In [11]:

# Step 7: Download backup without Google Drive

from google.colab import files
from pathlib import Path
import zipfile

filenames = [
    "swe015_dense_emnist.keras",
    "dense_results.json",
    "swe015_cnn_emnist.keras",
    "cnn_results.json"
]

zip_name = "SWE015_Models_Backup.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in filenames:
        path = Path("/content") / filename
        if path.is_file():
            z.write(path, arcname=filename)
            print("Added:", filename)
        else:
            print("MISSING:", filename)

print("Backup created!")
files.download(zip_name)


Added: swe015_dense_emnist.keras
Added: dense_results.json
Added: swe015_cnn_emnist.keras
Added: cnn_results.json
Backup created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>